<center>
    <h1>JaxTon</h1>
    <i>💯 JAX exercises</i>
    <br>
    <br>
    <a href='https://github.com/vopani/jaxton/blob/master/LICENSE'>
        <img src='https://img.shields.io/badge/license-Apache%202.0-blue.svg?logo=apache'>
    </a>
    <a href='https://github.com/vopani/jaxton'>
        <img src='https://img.shields.io/github/stars/vopani/jaxton?color=yellowgreen&logo=github'>
    </a>
    <a href='https://twitter.com/vopani'>
        <img src='https://img.shields.io/twitter/follow/vopani'>
    </a>
</center>

<center>
    This is Set 10: Capstone (Solutions 91-100) of <b>JaxTon</b>: <i>💯 JAX exercises</i>
    <br>
    You can find all the exercises and solutions on <a href="https://github.com/vopani/jaxton#exercises-">GitHub</a>
</center>

**Prerequisites**

* JAX should be installed via `uv sync` in the project root.
* This capstone set combines all previous concepts into an end-to-end project.

In [ ]:
import jax
import jax.numpy as jnp

In [ ]:
## generate a regression dataset: y = sin(x) + noise
key = jax.random.key(0)
k1, k2 = jax.random.split(key)
X_train = jax.random.uniform(k1, (200, 1), minval=-3, maxval=3)
y_train = jnp.sin(X_train) + 0.1 * jax.random.normal(k2, X_train.shape)

**Exercise 91: Initialize a 3-layer MLP with sizes [1, 32, 32, 1] using He initialization. Assign to `params`. (Reuse `init_mlp` or write from scratch.)**

In [ ]:
def init_layer(key, n_in, n_out):
    W_key, b_key = jax.random.split(key)
    return {"weights": jax.random.normal(W_key, (n_in, n_out)) * jnp.sqrt(2.0 / n_in),
            "bias": jnp.zeros(n_out)}

def init_mlp(key, sizes):
    params = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        key, subkey = jax.random.split(key)
        params.append(init_layer(subkey, n_in, n_out))
    return params

params = init_mlp(jax.random.key(42), [1, 32, 32, 1])
jax.tree.map(lambda x: x.shape, params)

**Exercise 92: Write a `forward(params, x)` function with `relu` hidden activations and linear output. Verify it runs on `X_train`.**

In [ ]:
def forward(params, x):
    *hidden, last = params
    for layer in hidden:
        x = jax.nn.relu(x @ layer["weights"] + layer["bias"])
    return x @ last["weights"] + last["bias"]

forward(params, X_train[:3])

**Exercise 93: Write an `mse_loss(params, x, y)` function. Compute the initial loss on the training data.**

In [ ]:
def mse_loss(params, x, y):
    return jnp.mean((forward(params, x) - y) ** 2)

mse_loss(params, X_train, y_train)

**Exercise 94: JIT-compile a `train_step(params, x, y, lr)` function that returns `(updated_params, loss)`. Assign to `jit_step`.**

In [ ]:
@jax.jit
def jit_step(params, x, y, lr):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y)
    params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
    return params, loss

jit_step(params, X_train, y_train, 0.01)

**Exercise 95: Train for 2000 steps with lr=0.01. Record the loss every 200 steps in a list `losses`.**

In [ ]:
params = init_mlp(jax.random.key(42), [1, 32, 32, 1])
losses = []
for step in range(2000):
    params, loss = jit_step(params, X_train, y_train, 0.01)
    if step % 200 == 0:
        losses.append(float(loss))
        print(f"Step {step}: {float(loss):.4f}")

**Exercise 96: Use `jax.vmap(forward, in_axes=(None, 0))` to produce predictions on a grid of 100 points from -3 to 3. Assign to `preds`.**

In [ ]:
x_grid = jnp.linspace(-3, 3, 100).reshape(-1, 1)
preds = jax.vmap(lambda x: forward(params, x.reshape(1, -1)).squeeze(), in_axes=0)(x_grid)
preds.shape

**Exercise 97: Use `jax.grad` to compute the derivative of the trained network's output with respect to its input (dy/dx). Evaluate on the same grid of 100 points. Assign to `derivs`.**

In [ ]:
grad_fn = jax.grad(lambda x: forward(params, x.reshape(1, -1)).squeeze())
derivs = jax.vmap(grad_fn)(x_grid)
derivs.shape

**Exercise 98: Use `jax.lax.scan` to rewrite the training loop as a single compiled function `scan_train(params, xs_ys, lr)` that returns final params and all losses. Run for 2000 steps.**

In [ ]:
def scan_train(params, xs_ys, lr):
    def step(params, _):
        loss, grads = jax.value_and_grad(mse_loss)(params, xs_ys[0], xs_ys[1])
        params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
        return params, loss
    params, losses = jax.lax.scan(step, params, None, length=2000)
    return params, losses

params_init = init_mlp(jax.random.key(42), [1, 32, 32, 1])
trained, all_losses = jax.jit(scan_train, static_argnums=2)(params_init, (X_train, y_train), 0.01)
print(f"Final loss: {float(all_losses[-1]):.4f}")

**Exercise 99: Compute the total number of parameters in your model using `jax.tree.leaves`. Then compute the L2 regularization term `lambda * sum(||w||^2)` over all weight matrices (not biases). Add it to the loss function as `reg_loss`.**

In [ ]:
total_params = sum(x.size for x in jax.tree.leaves(params))
print(f"Total parameters: {total_params}")

def reg_loss(params, x, y, lam=0.001):
    data_loss = mse_loss(params, x, y)
    l2 = sum(jnp.sum(layer["weights"] ** 2) for layer in params)
    return data_loss + lam * l2

reg_loss(params, X_train, y_train)

**Exercise 100: Put it all together: retrain from scratch using `reg_loss` with lambda=0.001, `jax.lax.scan`, and evaluate final predictions vs true sin(x). Print final loss and parameter count.**

In [ ]:
@jax.jit
def reg_step(params, x, y, lr):
    loss, grads = jax.value_and_grad(reg_loss)(params, x, y)
    params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
    return params, loss

params = init_mlp(jax.random.key(42), [1, 32, 32, 1])
for step in range(2000):
    params, loss = reg_step(params, X_train, y_train, 0.01)

total = sum(x.size for x in jax.tree.leaves(params))
preds = jax.vmap(lambda x: forward(params, x.reshape(1, -1)).squeeze())(x_grid)
print(f"Final loss: {float(loss):.4f}")
print(f"Parameters: {total}")
print(f"Predictions range: [{float(preds.min()):.2f}, {float(preds.max()):.2f}]")

<center>
    This completes Set 10: Capstone (Solutions 91-100) of <b>JaxTon</b>: <i>💯 JAX exercises</i>
    <br>
    You can find all the exercises and solutions on <a href="https://github.com/vopani/jaxton#exercises-">GitHub</a>
</center>